In [1]:
!pip install vaderSentiment

In [3]:
import pandas as pd

file_path = "Reviews.csv"
df = pd.read_csv(file_path, encoding="latin1")

# keep needed columns
df = df[['Score', 'Text']].dropna()

# reduce size (IMPORTANT so it runs fast)
df = df.sample(n=20000, random_state=42)

# create sentiment labels
df['Label'] = df['Score'].apply(
    lambda x: 'negative' if x in [1, 2] else 'neutral' if x == 3 else 'positive'
)

print(df.head())
print(df['Label'].value_counts())

        Score                                               Text     Label
165256      5  Having tried a couple of other brands of glute...  positive
231465      5  My cat loves these treats. If ever I can't fin...  positive
427827      3  A little less than I expected.  It tends to ha...   neutral
433954      2  First there was Frosted Mini-Wheats, in origin...  negative
70260       5  and I want to congratulate the graphic artist ...  positive
Label
positive    15595
negative     2886
neutral      1519
Name: count, dtype: int64


In [4]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = analyzer.polarity_scores(text)['compound']
    
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['VADER_Pred'] = df['Text'].apply(get_sentiment)

print(df[['Text', 'Label', 'VADER_Pred']].head())

                                                     Text     Label VADER_Pred
165256  Having tried a couple of other brands of glute...  positive   positive
231465  My cat loves these treats. If ever I can't fin...  positive   positive
427827  A little less than I expected.  It tends to ha...   neutral   positive
433954  First there was Frosted Mini-Wheats, in origin...  negative   positive
70260   and I want to congratulate the graphic artist ...  positive   positive


In [5]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

vader_acc = accuracy_score(df['Label'], df['VADER_Pred'])

print("=== VADER Lexicon Model ===")
print("Accuracy:", vader_acc)
print(classification_report(df['Label'], df['VADER_Pred']))
print("Confusion Matrix:\n", confusion_matrix(df['Label'], df['VADER_Pred']))

=== VADER Lexicon Model ===
Accuracy: 0.8051
              precision    recall  f1-score   support

    negative       0.60      0.41      0.49      2886
     neutral       0.15      0.04      0.06      1519
    positive       0.84      0.95      0.89     15595

    accuracy                           0.81     20000
   macro avg       0.53      0.47      0.48     20000
weighted avg       0.75      0.81      0.77     20000

Confusion Matrix:
 [[ 1175   130  1581]
 [  253    58  1208]
 [  520   206 14869]]
